# Reinforcement Learning

O curso usa a gym mas ela foi descontinuada entao eu troquei pela gymnasium

In [1]:
import gymnasium as gym
import random

random.seed(1234)

# monto o mapa do taxi em modo texto
streets = gym.make("Taxi-v3",render_mode="ansi").env
streets.reset()

print("\n" + streets.render())


+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




In [2]:
# forco um ponto de partida fixo pra eu conseguir comparar depois
initial_state = streets.unwrapped.encode(2, 3, 2, 0)

streets.unwrapped.s = initial_state

print(streets.render())

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




In [3]:
streets.unwrapped.P[initial_state] # as acoes possiveis daqui e o que cada uma rende

{0: [(1.0, 368, -1, False)],
 1: [(1.0, 168, -1, False)],
 2: [(1.0, 288, -1, False)],
 3: [(1.0, 248, -1, False)],
 4: [(1.0, 268, -10, False)],
 5: [(1.0, 268, -10, False)]}

In [4]:
import gymnasium as gym
import numpy as np
import random

streets = gym.make("Taxi-v3", render_mode="ansi").env

# tabela com todo estado na linha e toda acao na coluna
q_table = np.zeros([streets.observation_space.n, streets.action_space.n])

learning_rate = 0.1   # o quanto ele aprende por passo
discount_factor = 0.6 # o quanto ele liga pra recompensa la na frente
exploration = 0.1     # a chance dele tentar algo aleatorio
epochs = 10000        # quantas viagens de treino

for taxi_run in range(epochs):
    state, info = streets.reset()
    done = False

    while not done:
        random_value = random.uniform(0, 1)
        if(random_value < exploration):
            action = streets.action_space.sample() # tenta uma coisa nova
        else:
            action = np.argmax(q_table[state])     # usa o que ja deu certo

        next_state, reward, terminated, truncated, info = streets.step(action)
        done = terminated or truncated
        prev_q = q_table[state, action]
        next_max_q = np.max(q_table[next_state])

        # essa conta aqui e a equacao de Bellman
        new_q = (1 - learning_rate) * prev_q + learning_rate * (reward + discount_factor * next_max_q)
        q_table[state, action] = new_q

        state = next_state

In [5]:
initial_state, info = streets.reset()
print(q_table[initial_state]) # agora cada acao tem uma nota e a maior e a melhor

[-2.48296545 -2.48228565 -2.48266905 -2.48274632 -8.42355666 -8.40582743]


In [ ]:
from IPython.display import clear_output
from time import sleep
import numpy as np

# agora eu so uso o que foi treinado sem explorar mais nada
for tripnum in range(1, 11):

    state, info = streets.reset()
    done = False

    while not done:
        action = np.argmax(q_table[state])
        next_state, reward, terminated, truncated, info = streets.step(action)
        done = terminated or truncated

        clear_output(wait=True)
        print("trip number " + str(tripnum))

        print(streets.render())

        sleep(.5)
        state = next_state
    sleep(2)

trip number 4
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (West)

